In [1]:
from tensorflow import keras
from tensorflow.keras import layers

###(1) Residual connections
The residual connections act as an information shortcut around destructive or noisy blocks (such as blocks that contain relu activation or dropoutslayers), enabling error gradient information from early layers to propagate noiselessly through the deep network. so the last layer contain not just the processes information through the layers but also the original information of the input layer.

lets look at an example of simple convnet structured into series of blocks, each made of two convolution layers and one optional max pooling with a residual connection around each block.

In [2]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Rescaling(1./255)(inputs)

#utility function below to apply a convolutional block with a residual connection with an option to add max pooling
def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
    x = layers.Conv2D(filters, 3, activation='relu', padding='same')(x)

    if pooling:
      x = layers.MaxPooling2D(2, padding='same')(x)
      residual = layers.Conv2D(filters, 1, strides=2)(residual) # if we use maxpooling, we add a strided convolution to..
                                                                #.. project the residual to the expected shape.
    elif filters != residual.shape[-1]:
      residual = layers.Conv2D(filters, 1)(residual)  #if we don't use max pooling, we only project the residuals if the..
                                                      #..number of channels has changed
    x = layers.add([x, residual])
    return x

In [3]:
x = residual_block(x, filters=32, pooling=True)  # first block
x = residual_block(x, filters=64, pooling=True)  # second block; note the increasing filter count in each  block
x = residual_block(x, filters=128, pooling=True) # last block does'nt need a max pooling layer since we will apply global..
                                                #..average pooling right after it.
x = layers.GlobalAveragePooling2D()(x)          # This computes the average of all the channels, instead of just flattening it.
output = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        896 │ rescaling[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 16,    │        128 │ rescaling[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 16, 16,    │          0 │ max_pooling2d[0]… │
│                     │ 32)               │            │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │     18,496 │ add[0][0]         │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │     36,928 │ conv2d_3[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 8, 8, 64)  │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 8, 8, 64)  │      2,112 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 8, 8, 64)  │          0 │ max_pooling2d_1[… │
│                     │                   │            │ conv2d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 8, 8, 128) │     73,856 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 8, 8, 128) │    147,584 │ conv2d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 4, 4, 128) │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 4, 4, 128) │      8,320 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 4, 4, 128) │          0 │ max_pooling2d_2[… │
│                     │                   │            │ conv2d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ add_2[0][0]       │
│ (GlobalAveragePool… │                   │            │                 

 Total params: 297,697 (1.14 MB)

 Trainable params: 297,697 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

###(2) Batch Normalization
Normalization is a broad category of methods that seek to make different samples seen by a machine learning model more similar to each other, which helps th model learn and generalize well to new data.
 a) Batch  Normalization: is a type of layer that can adaptively normalize data even as the mean and variance change over time during training. This makes sure that the data entering any layer is duely normalized.

 Batch normalization, just as residual connections helps with gradient propagation, and thus allows for deeper networks. It can be used after any layer - Dense, Conv2D etc. :

In [6]:
input_layer = layers.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, use_bias=False)(input_layer)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

It is recommended to use the activation after the BatchNormalization, like the following:


In [7]:
x = layers.Conv2D(32, 3,use_bias=False)(input_layer)  # NOte the lack of activation here
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)   # Activation is placed after the normalization layer

Its worth noting that when using batch normalization, there is no need for using bias as seen from the previous example (where use_bias = False), this is because the normalization takes care of centering the layers output on zero.

### (3) Depthwise seperable convolutions